# FSC-LEGACY-YES-001 Blueprint unit economics - revised
All prices, seats, host terms, reach, conversion, refunds, support, labor values, and cannibalization are planning assumptions, not observed demand. The event grid varies price and seats independently. The portfolio model fixes qualified audience reach and applies choice/cannibalization drag instead of multiplying sales per SKU. No sale, payment, outreach, or live account action occurred.

In [ ]:
from math import ceil

USD_PER_BHD = 2.659
LABOR_BHD_PER_HOUR = 5.0  # shadow value, not cash paid to agents
PAYMENT_RATE = .03
REFUND_RESERVE_RATE = .05
OPS_HOURS = 2.0  # conservative first-of-kind case
FIXED_HOST_BHD = 30.0

def event_contribution(seats, price_bhd, host_bhd=FIXED_HOST_BHD):
    gross = seats * price_bhd
    net_after_fees_refunds = gross * (1 - PAYMENT_RATE - REFUND_RESERVE_RATE)
    ops_labor = OPS_HOURS * LABOR_BHD_PER_HOUR
    return round(net_after_fees_refunds - host_bhd - ops_labor, 2)

event_surface_fixed_host = {
    price: {seats: event_contribution(seats, price) for seats in (4, 5, 6, 8, 10)}
    for price in (5, 7, 9)
}
host_break_even_ceiling = {
    price: {seats: round(seats * price * (1-PAYMENT_RATE-REFUND_RESERVE_RATE) - OPS_HOURS*LABOR_BHD_PER_HOUR, 2) for seats in (4,5,6,8,10)}
    for price in (5,7,9)
}
revenue_share_70_surface = {
    price: {seats: round(seats*price*(1-PAYMENT_RATE-REFUND_RESERVE_RATE-.70)-OPS_HOURS*LABOR_BHD_PER_HOUR, 2) for seats in (4,5,6,8,10)}
    for price in (5,7,9)
}

sku_price_usd = 12.0
gumroad_direct_fee_usd = sku_price_usd*.10 + .50 + sku_price_usd*.029 + .30
refund_reserve_usd = sku_price_usd*.05
support_labor_usd = (5/60)*LABOR_BHD_PER_HOUR*USD_PER_BHD
sku_contribution_usd = sku_price_usd-gumroad_direct_fee_usd-refund_reserve_usd-support_labor_usd
sku_contribution_bhd = sku_contribution_usd/USD_PER_BHD
first_sku_cost_bhd = 12*LABOR_BHD_PER_HOUR+20
follow_on_cost_bhd = 4*LABOR_BHD_PER_HOUR+5

audience_scenarios = {
    'conservative': dict(qualified_visits=250, conversion=.01),
    'base': dict(qualified_visits=500, conversion=.02),
    'upside': dict(qualified_visits=1500, conversion=.03),
}
def choice_factor(sku_count):
    return max(.50, 1-.02*(sku_count-1))  # explicit unverified choice/cannibalization assumption

portfolio = {}
for sku_count in (7,10,25,50,100):
    build_cost = first_sku_cost_bhd+(sku_count-1)*follow_on_cost_bhd
    portfolio[sku_count] = {}
    for name,case in audience_scenarios.items():
        total_sales = case['qualified_visits']*case['conversion']*choice_factor(sku_count)
        contribution = total_sales*sku_contribution_bhd
        portfolio[sku_count][name] = {
            'qualified_visits_catalog': case['qualified_visits'],
            'conversion_assumption': case['conversion'],
            'choice_factor': round(choice_factor(sku_count),2),
            'total_sales_catalog': round(total_sales,2),
            'sales_per_sku': round(total_sales/sku_count,2),
            'net_after_build_bhd': round(contribution-build_cost,2),
        }

results = {
    'event_surface_fixed_bhd30_host_after_2h_ops': event_surface_fixed_host,
    'maximum_break_even_host_compensation_bhd': host_break_even_ceiling,
    'event_surface_70pct_host_share_after_2h_ops': revenue_share_70_surface,
    'sku_contribution_usd': round(sku_contribution_usd,2),
    'sku_contribution_bhd': round(sku_contribution_bhd,2),
    'first_sku_break_even_sales': ceil(first_sku_cost_bhd/sku_contribution_bhd),
    'follow_on_break_even_sales': ceil(follow_on_cost_bhd/sku_contribution_bhd),
    'reach_limited_portfolio': portfolio,
}
results